In [1]:
import os
from pathlib import Path

import pandas as pd
import time


def get_env_value(key, default=None, env_path=".env"):
    # Prefer shell variables first, then fall back to the local .env file so the notebook works in VS Code and headless runs.
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default


def get_env_int(key, default):
    return int(get_env_value(key, str(default)))


def get_env_float(key, default):
    return float(get_env_value(key, str(default)))


# Keep the notebook data paths configurable so the same code works across local machines and test setups.
csv_path = Path(get_env_value("CSV_PATH", "IOT Data Simulation/smart_logistic_tracker_japan.csv"))
sample_rows = get_env_int("SAMPLE_ROWS", 5)
write_delay_seconds = get_env_float("WRITE_DELAY_SECONDS", 0.1)
write_gas_limit = get_env_int("WRITE_GAS_LIMIT", 3000000)
enable_duplicate_writes = get_env_value("ENABLE_DUPLICATE_WRITES", "false").lower() in {"1", "true", "yes", "on"}
abi_path = Path(get_env_value("ABI_PATH", "contracts/abi.json"))

# Load the CSV file with basic error handling
try:
    df = pd.read_csv(csv_path)
    print(f"Total records in CSV: {len(df)}")
    print(f"First {sample_rows} records:")

    # Display the first few rows
    display(df.head(sample_rows))
except FileNotFoundError:
    print(f"❌ CSV file not found: {csv_path}")
    df = pd.DataFrame()
except pd.errors.EmptyDataError:
    print(f"❌ CSV file is empty: {csv_path}")
    df = pd.DataFrame()
except pd.errors.ParserError as error:
    print(f"❌ Failed to parse CSV file {csv_path}: {error}")
    df = pd.DataFrame()
except Exception as error:
    print(f"❌ Unexpected error while loading {csv_path}: {error}")
    df = pd.DataFrame()

Total records in CSV: 100
First 5 records:


,timestamp,asset_id,shipment_id,latitude,longitude,shipment_status,inventory_level,waiting_time,temperature,humidity,traffic_status,demand_forecast,asset_utilization,logistics_delay_reason,logistics_delay,rfid_tag,rfid_verified,tamper_alert
0,2026-05-04 15:42:12.730007,TRUCK344,SHP4147,35.353926,139.411921,Out for Delivery,238,75,20.8,85,Clear,184,54.26,Mechanical Failure,0,RFID208288,False,No
1,2026-05-04 10:09:12.731395,TRUCK412,SHP6541,35.348310,139.095461,In Transit,82,33,17.3,85,Detour,226,90.57,Traffic,1,RFID252723,True,Yes
2,2026-05-04 11:46:12.731581,TRUCK830,SHP8910,35.844433,139.798511,Out for Delivery,474,106,8.3,63,Heavy,152,62.72,NaN,1,RFID485390,False,No
3,2026-05-04 01:44:12.731706,TRUCK878,SHP8090,35.787743,139.869721,Out for Delivery,452,85,1.0,53,Heavy,138,87.25,Traffic,0,RFID973827,False,Yes
4,2026-05-04 19:53:12.731804,TRUCK197,SHP5767,35.148404,139.187463,In Transit,333,97,-0.2,32,Heavy,235,93.44,Traffic,1,RFID101636,True,No


In [2]:
from web3 import Web3

# Connect to local blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Verify connection
if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [3]:
import json

# Use the loaded ABI path and deployed contract address.
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

# Load the ABI that matches the deployed contract in this repository.
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Load the smart contract.
contract = web3.eth.contract(address=contract_address, abi=abi)

# Ganache may expose a different unlocked account set than the deployed contract owner,
# so we fall back to an explicit override when needed.
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if not override_owner:
        raise ValueError(
            f"Contract owner {contract_owner} is not unlocked in Ganache. "
            "Set CONTRACT_OWNER in .env to an unlocked account."
        )
    contract_owner = Web3.to_checksum_address(override_owner)
    if contract_owner not in web3.eth.accounts:
        raise ValueError(
            f"CONTRACT_OWNER {contract_owner} is not unlocked in Ganache."
        )

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x7abf4b356FB67C8a9917c7E1E543895DB1Bf53b4
✅ Using sender account: 0x1C73Dd704ffeE88a4f4aAD5bA3B1af87C5884D0F


In [4]:
# Retrieve all existing records from the blockchain once at startup to cache them.
# This prevents expensive O(N) blockchain roundtrips for duplicate verification on every iteration.
total_records = contract.functions.getTotalRecords().call()
existing_records = set()
for record_index in range(total_records):
    record = contract.functions.getRecord(record_index).call()
    existing_records.add((str(record[1]), str(record[2]), str(record[3])))

def record_exists(package_id, data_type, data_value):
    """Return True when the exact record is already in the cache."""
    return (str(package_id), str(data_type), str(data_value)) in existing_records


def send_iot_data(package_id, data_type, data_value):
    """
    Sends logistics IoT data
    to the deployed smart contract
    """

    # Skip exact duplicates unless testing explicitly requires them.
    if not enable_duplicate_writes and record_exists(package_id, data_type, data_value):
        print(
            f"ℹ️ Skipped duplicate | {package_id} | "
            f"Type: {data_type} | Value: {data_value}"
        )
        return False

    if enable_duplicate_writes:
        print("ℹ️ Duplicate-write mode is ON; exact duplicates will be stored.")

    txn = contract.functions.storeData(
        package_id,
        data_type,
        data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': write_gas_limit
    })

    # Wait for transaction confirmation before moving to the next record.
    receipt = web3.eth.wait_for_transaction_receipt(txn)

    # Cache the new entry locally
    existing_records.add((str(package_id), str(data_type), str(data_value)))

    print(
        f"✅ Data Stored | {package_id} | "
        f"Type: {data_type} | "
        f"Value: {data_value} | "
        f"Txn Hash: {receipt.transactionHash.hex()}"
    )
    return True

# Each CSV row writes multiple contract entries depending on the columns.
is_iot_data = "shipment_id" in df.columns
entries_per_row = 3 if is_iot_data else 2

target_contract_records = int(get_env_value("TARGET_CONTRACT_RECORDS", "100"))
target_rows = target_contract_records // entries_per_row
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records
rows_to_store = min(len(df), target_rows, remaining_entries // entries_per_row)

print(f"Target contract records: {target_contract_records}")
print(f"Current records: {current_records}")
print(f"Maximum records: {max_entries}")
print(f"Remaining contract slots: {remaining_entries}")
print(f"Rows that can still be stored safely: {rows_to_store}")
print(f"Duplicate writes enabled: {enable_duplicate_writes}")

if target_contract_records % entries_per_row != 0:
    print(f"⚠️ TARGET_CONTRACT_RECORDS is not a multiple of {entries_per_row}, ignoring remaining slots to keep rows complete.")

if rows_to_store <= 0:
    print("⚠️ No remaining storage capacity on the contract.")
else:
    stored_rows = 0
    skipped_rows = 0

    for index, row in df.head(rows_to_store).iterrows():
        if is_iot_data:
            package_id = str(row["shipment_id"])
            status = str(row["shipment_status"])
            temp = f"{row['temperature']}°C"
            humid = f"{row['humidity']}%"

            s1 = send_iot_data(package_id, "Status", status)
            s2 = send_iot_data(package_id, "Temperature", temp)
            s3 = send_iot_data(package_id, "Humidity", humid)

            if s1 or s2 or s3:
                stored_rows += 1
            else:
                skipped_rows += 1
        else:
            package_id = str(row["package_id"])
            location = str(row["current_location"])
            status = str(row["latest_status"])

            location_stored = send_iot_data(package_id, "Location", location)
            status_stored = send_iot_data(package_id, "Status", status)

            if location_stored and status_stored:
                stored_rows += 1
            else:
                skipped_rows += 1

        # Small pause between transactions keeps Ganache logs readable and avoids flooding the provider.
        time.sleep(write_delay_seconds)

    print(f"\n✅ Successfully stored {stored_rows} new rows on the blockchain!")
    if skipped_rows:
        print(f"ℹ️ Skipped {skipped_rows} duplicate rows.")

Target contract records: 100
Current records: 242
Maximum records: 500
Remaining contract slots: 258
Rows that can still be stored safely: 33
Duplicate writes enabled: False
⚠️ TARGET_CONTRACT_RECORDS is not a multiple of 3, ignoring remaining slots to keep rows complete.
ℹ️ Skipped duplicate | SHP4147 | Type: Status | Value: Out for Delivery
ℹ️ Skipped duplicate | SHP4147 | Type: Temperature | Value: 20.8°C
ℹ️ Skipped duplicate | SHP4147 | Type: Humidity | Value: 85%
ℹ️ Skipped duplicate | SHP6541 | Type: Status | Value: In Transit
ℹ️ Skipped duplicate | SHP6541 | Type: Temperature | Value: 17.3°C
ℹ️ Skipped duplicate | SHP6541 | Type: Humidity | Value: 85%


ℹ️ Skipped duplicate | SHP8910 | Type: Status | Value: Out for Delivery
ℹ️ Skipped duplicate | SHP8910 | Type: Temperature | Value: 8.3°C
ℹ️ Skipped duplicate | SHP8910 | Type: Humidity | Value: 63%
ℹ️ Skipped duplicate | SHP8090 | Type: Status | Value: Out for Delivery
ℹ️ Skipped duplicate | SHP8090 | Type: Temperature | Value: 1.0°C
ℹ️ Skipped duplicate | SHP8090 | Type: Humidity | Value: 53%


ℹ️ Skipped duplicate | SHP5767 | Type: Status | Value: In Transit
ℹ️ Skipped duplicate | SHP5767 | Type: Temperature | Value: -0.2°C
ℹ️ Skipped duplicate | SHP5767 | Type: Humidity | Value: 32%
ℹ️ Skipped duplicate | SHP2407 | Type: Status | Value: In Transit
ℹ️ Skipped duplicate | SHP2407 | Type: Temperature | Value: 16.4°C
ℹ️ Skipped duplicate | SHP2407 | Type: Humidity | Value: 38%


ℹ️ Skipped duplicate | SHP6770 | Type: Status | Value: Delayed
ℹ️ Skipped duplicate | SHP6770 | Type: Temperature | Value: 0.5°C
ℹ️ Skipped duplicate | SHP6770 | Type: Humidity | Value: 51%
ℹ️ Skipped duplicate | SHP4992 | Type: Status | Value: Out for Delivery
ℹ️ Skipped duplicate | SHP4992 | Type: Temperature | Value: 16.5°C
ℹ️ Skipped duplicate | SHP4992 | Type: Humidity | Value: 63%


ℹ️ Skipped duplicate | SHP8087 | Type: Status | Value: In Transit
ℹ️ Skipped duplicate | SHP8087 | Type: Temperature | Value: -0.9°C
ℹ️ Skipped duplicate | SHP8087 | Type: Humidity | Value: 51%
ℹ️ Skipped duplicate | SHP7190 | Type: Status | Value: Delayed
ℹ️ Skipped duplicate | SHP7190 | Type: Temperature | Value: 1.1°C
ℹ️ Skipped duplicate | SHP7190 | Type: Humidity | Value: 63%


...

✅ Data Stored | SHP2076 | Type: Temperature | Value: 13.5°C | Txn Hash: 375eba1fdab162d46df867e97c603e5db731ef3c3c575a7774eb2212ebe1bd0b
✅ Data Stored | SHP2076 | Type: Humidity | Value: 84% | Txn Hash: 72e67bfdc036de589f6af210a126b58b13a213c0d0099235525b831030398b5b
✅ Data Stored | SHP6669 | Type: Status | Value: Delayed | Txn Hash: e913cf8a9e107120da8cfd118cde4a526c351fb98b1bc3d9867ceda2ac9156a4
✅ Data Stored | SHP6669 | Type: Temperature | Value: 5.7°C | Txn Hash: 53e4bb245c994cc665ed8d7ef38e6a9caccfb999cd6a3390617a00d552154a1c
✅ Data Stored | SHP6669 | Type: Humidity | Value: 89% | Txn Hash: f3590509e288a4efa6753f7e0f3ac47e7c9e35d92e134a81abbe995c5500912b


✅ Data Stored | SHP6849 | Type: Status | Value: Delivered | Txn Hash: 91cd3ddcebdc5a72ebe433ec57dbd08befb380c08fa5721750993de79583cb84
✅ Data Stored | SHP6849 | Type: Temperature | Value: 11.9°C | Txn Hash: 0ae9e41ef3f48b748b8351b7086eda646ca46dd001624ed7cba671f9d205c7dc
✅ Data Stored | SHP6849 | Type: Humidity | Value: 87% | Txn Hash: bcedaf443cc7a2c607d4b10c592be067a1beb0dc50da870ba7b5a6c293abd59a


✅ Data Stored | SHP9026 | Type: Status | Value: In Transit | Txn Hash: 618ef4bd4033f151e041597e7b2f3a2b1612d2c4c6ae4d18339d01fe89a13abc
✅ Data Stored | SHP9026 | Type: Temperature | Value: -1.6°C | Txn Hash: 1ca866838cb28bd17b3cfd7bf2559f189f3c2eaaefdd858a82cce54833bb6a76
✅ Data Stored | SHP9026 | Type: Humidity | Value: 37% | Txn Hash: 0a15bf2b5ad251f8b450d83291b8ee302501ba65dbad5b12217714d9aadb1b2a
✅ Data Stored | SHP4292 | Type: Status | Value: In Transit | Txn Hash: f0081d9902185c6f44c5276195f6b28549e8bb4b3a73d6496fb37deb0edc7638
✅ Data Stored | SHP4292 | Type: Temperature | Value: 23.7°C | Txn Hash: 6f6dd49efc1eff657eff4a3f40ef7fedbb919ae5c4bcfd9dd4368074dc0bf3a8


✅ Data Stored | SHP4292 | Type: Humidity | Value: 82% | Txn Hash: 574a9cf4627e090a85b4f3afc05548e358e329b493f20bf149c31d0969493d89
✅ Data Stored | SHP7754 | Type: Status | Value: Delayed | Txn Hash: c9b50bdc097a16814d7bb46401a73d510895780fd9179379236d346131f8cc27
✅ Data Stored | SHP7754 | Type: Temperature | Value: -2.8°C | Txn Hash: b6d9649322880e6acd05a12709b9f0d286b822e9fe69cb898ec66e51d7cc4303
✅ Data Stored | SHP7754 | Type: Humidity | Value: 73% | Txn Hash: 618af142e6e41464241eda21ef39728b3d15c1813efd504a687040a31c3e4c98



✅ Successfully stored 8 new rows on the blockchain!
ℹ️ Skipped 25 duplicate rows.


In [5]:
current_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {current_records}")

Total IoT records stored: 265


In [6]:
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records

print(f"Current IoT records stored: {current_records}")
print(f"Maximum records allowed: {max_entries}")
print(f"Remaining storage slots: {remaining_entries}")

if remaining_entries == 0:
    print("⚠️ The contract is full. Redeploy a new contract or reset the chain to store more data.")
elif remaining_entries <= 20:
    print("⚠️ The contract is nearing capacity.")

Current IoT records stored: 265
Maximum records allowed: 500
Remaining storage slots: 235


In [7]:
# Retrieve and display the first stored record
first_record = contract.functions.getRecord(0).call()

print("📦 First Stored Record")
print(f"Timestamp: {first_record[0]}")
print(f"Package ID: {first_record[1]}")
print(f"Data Type: {first_record[2]}")
print(f"Data Value: {first_record[3]}")

📦 First Stored Record
Timestamp: 1780192485
Package ID: PKG7545
Data Type: Location
Data Value: Naha Central Post Office
